# Colab GPU: same patient-conditioned experiment

First run local audit/preprocess, then `python src/report.py --bundle --config configs/pilot.yaml`. Manually put ONLY `outputs/pilot/colab_bundle.zip` in `MyDrive/ptbxl_assignment/`. The bundle excludes the reference PDF, raw dataset, and checkpoints. Select **Runtime > Change runtime type > T4 GPU**. This notebook does not upload files automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
drive_dir = Path('/content/drive/MyDrive/ptbxl_assignment')
project = Path('/content/ptbxl_assignment')
project.mkdir(exist_ok=True)
with ZipFile(drive_dir / 'colab_bundle.zip') as archive:
    assert all(not n.lower().endswith('.pdf') and '..' not in Path(n).parts and not Path(n).is_absolute() for n in archive.namelist())
    archive.extractall(project)
os.chdir(project)
import sys
sys.path.insert(0, str((project / 'src').resolve()))

In [ ]:
%pip install -q -r requirements.txt
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before full training.'

## Persist outputs and checkpoints

Keep arrays on Colab local disk for speed, but save artifacts to Drive each epoch. Re-running the same configuration requires --resume to continue a saved checkpoint. CPU/GPU changes can affect numerical behavior; exact cross-device reproducibility is not guaranteed.

In [ ]:
import yaml
from config import load_config
cfg = load_config('configs/pilot.yaml')
cfg['data']['root'] = 'PTB-XL'  # raw files not needed for a validated cache
cfg['data']['cache'] = 'data/pilot'
cfg['training']['device'] = 'auto'
for key in ('experiments','outputs','documents'):
    cfg['paths'][key] = str(drive_dir / 'pilot' / key)
Path('configs/colab.yaml').write_text(yaml.safe_dump(cfg))
from preprocessing import validate_cache
validate_cache(cfg)['fingerprint']

## Run one step at a time

Choose the model and variant in each command. Use --resume on train.py only when continuing an existing run. Repeat training and evaluation manually for each desired model/variant.

In [ ]:
!python src/train.py --config configs/colab.yaml --model lstm --variant patient_history

In [ ]:
!python src/evaluate.py --config configs/colab.yaml --model lstm --variant patient_history

In [ ]:
!python src/evaluate.py --baselines --config configs/colab.yaml

In [ ]:
!python src/compare.py --config configs/colab.yaml

In [ ]:
!python src/report.py --config configs/colab.yaml

In [ ]:
import pandas as pd
pd.read_csv(drive_dir / 'pilot/outputs/model_comparison.csv')

## Return results to the local project

Download `experiments`, `outputs`, and `documents` from this dedicated Drive folder. Place documents under local `Project_Documents`, retain experiment names, then run `python src/report.py --config configs/pilot.yaml`. Do not mix fingerprints. Keep existing CPU results separately if retaining both CPU and GPU results.